<a href="https://colab.research.google.com/github/ChaimElchik/GPS-Demo/blob/main/GPS_DEM_DepthAnythingDistanceSamplingV3WIP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Elephant Detection and GPS Localization Demo 🐘📍

Welcome! This Colab notebook allows you to upload a single drone image, detect elephants within it using a YOLO model, and estimate their GPS coordinates.

**How to use:**
1.  Run the "Setup" cell to install necessary libraries and download the model.
2.  Run the "Upload Image" cell and select a JPG image from your computer.
3.  Run the "Process Image and Get Results" cell. This will perform detection, calculate GPS, and display the results.
4.  The results (image with IDs, coordinates CSV, distances CSV) will be displayed and made available for download.

---
## Setup Environment

In [15]:
# Install necessary packages
!pip install piexif geopy pyproj torch torchvision transformers timm accelerate -q
!pip install ultralytics==8.3.18 --upgrade --quiet
!apt-get install -y exiftool -qq

# Import necessary libraries
import cv2
import torch
import numpy as np
from ultralytics import YOLO
import pandas as pd
import time
import os
from pathlib import Path
import matplotlib.pyplot as plt
from pyproj import Transformer
import math
import re
from datetime import datetime, timedelta
import subprocess
import json
import csv
from google.colab import files
from PIL import Image
import io
from geopy.distance import geodesic
from IPython.display import Image as IPImage, display

# Create output directories
os.makedirs("Detections", exist_ok=True)
os.makedirs("Processed_Output", exist_ok=True)

print("\nSetup Complete!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 876.6/876.6 kB 15.3 MB/s eta 0:00:00

Setup Complete!



##  1. Upload Model, SRT File, Video File and Tracker Config File

Download the Model from https://github.com/ChaimElchik/GPS-Demo/blob/main/best_xl.pt

In [4]:
from google.colab import files
import os

print("--- Step 1: Upload Files ---")
print("Please upload your video, SRT file, your .pt model, and your tracker config (e.g., 'botsort.yaml').")

# Upload files
uploaded = files.upload()

# Store paths to the uploaded files (they are in the root directory)
video_path = None
srt_path = None
model_path = None
tracker_config_path = None

for fn in uploaded.keys():
    if fn.lower().endswith('.pt'):
        model_path = fn
        print(f"✅ Model file '{fn}' found.")
    elif fn.lower().endswith('.yaml'):
        tracker_config_path = fn
        print(f"✅ Tracker config file '{fn}' found.")
    elif fn.lower().endswith(('.mp4', '.mov', '.avi')):
        video_path = fn
        print(f"✅ Video file '{fn}' found.")
    elif fn.lower().endswith('.srt'):
        srt_path = fn
        print(f"✅ SRT file '{fn}' found.")

# --- Verification ---
print("\n--- Verifying files ---")
if not video_path:
    print("❌ ERROR: Video file not uploaded.")
if not srt_path:
    print("❌ ERROR: SRT file not uploaded.")
if not model_path:
     print(f"❌ ERROR: Model .pt file not uploaded.")
if not tracker_config_path:
    print("❌ ERROR: Tracker config .yaml file not uploaded.")

if video_path and srt_path and model_path and tracker_config_path:
    # --- IMPORTANT: Set the global MODEL_PATH variable for the next cell ---
    # This ensures the next cell knows which .pt file to use
    MODEL_PATH = model_path
    print("\n--- All files ready for processing! ---")


--- Step 1: Upload Files ---
Please upload your video, SRT file, your .pt model, and your tracker config (e.g., 'botsort.yaml').


Saving best.pt to best (1).pt
Saving botsortV5.yaml to botsortV5 (1).yaml
Saving DJI_20250618120033_0001_D.MP4 to DJI_20250618120033_0001_D (1).MP4
Saving DJI_20250618120033_0001_D.SRT to DJI_20250618120033_0001_D (2).SRT
✅ Model file 'best (1).pt' found.
✅ Tracker config file 'botsortV5 (1).yaml' found.
✅ Video file 'DJI_20250618120033_0001_D (1).MP4' found.
✅ SRT file 'DJI_20250618120033_0001_D (2).SRT' found.

--- Verifying files ---

--- All files ready for processing! ---


---
## 2. Core Logic and Helper Functions


In [12]:
# --- Import Libraries ---
import os
import cv2
import numpy as np
import subprocess
import json
import re
import math
import time
import traceback
import csv
from pathlib import Path
import requests
import torch
from transformers import AutoImageProcessor, AutoModelForDepthEstimation
from PIL import Image
import shutil

# --- Dependencies Check ---
try:
    from ultralytics import YOLO
    from geopy.distance import geodesic
    from geopy.point import Point
except ImportError as e:
    print(f"ERROR: Missing dependency - {e}. Please install required libraries.")
    print("Run: pip install ultralytics opencv-python pyproj geopy requests torch torchvision transformers timm accelerate Pillow")
    exit()

# --- Global Configuration ---
OUTPUT_DIR = "Video_Processing_Output"
# MODEL_PATH is now set dynamically in the previous cell and is no longer hardcoded here.
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEPTH_MODEL_NAME = 'depth-anything/Depth-Anything-V2-Large-hf'

# --- SENSOR CONFIGURATION ---
SENSOR_WIDTH_MM = 17.3
SENSOR_HEIGHT_MM = 13.0
print(f"INFO: Using Sensor Size {SENSOR_WIDTH_MM}mm x {SENSOR_HEIGHT_MM} (Mavic 3 Pro Main Cam).")
print(f"INFO: Using device: {DEVICE} for deep learning models.")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- SRT Parsing Function ---
def parse_srt_file(srt_path):
    """Parses the custom SRT file to extract metadata for each frame."""
    print(f"INFO: Parsing SRT file: {srt_path}")
    metadata_map = {}
    with open(srt_path, 'r') as f:
        content = f.read()

    pattern = re.compile(
        r"FrameCnt: (\d+).*?"
        r"\[focal_len: ([\d\.]+)\]"
        r".*?\[latitude: ([\d\.\-]+)\] \[longitude: ([\d\.\-]+)\] "
        r"\[rel_alt: ([\d\.\-]+) abs_alt: ([\d\.\-]+)\] "
        r"\[gb_yaw: ([\d\.\-]+) gb_pitch: ([\d\.\-]+) gb_roll: ([\d\.\-]+)\]",
        re.DOTALL
    )

    matches = pattern.finditer(content)
    for match in matches:
        frame_cnt = int(match.group(1))
        metadata_map[frame_cnt] = {
            'focal_len': float(match.group(2)), 'latitude': float(match.group(3)),
            'longitude': float(match.group(4)), 'rel_alt': float(match.group(5)),
            'abs_alt': float(match.group(6)), 'gb_yaw': float(match.group(7)),
            'gb_pitch': float(match.group(8)), 'gb_roll': float(match.group(9)),
        }
    print(f"✅ Successfully parsed metadata for {len(metadata_map)} frames from SRT.")
    return metadata_map

# --- Depth Map and Geolocation Functions (Unchanged) ---
def get_depth_map(frame, model, processor):
    image = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    inputs = processor(images=image, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        outputs = model(**inputs)
        predicted_depth = outputs.predicted_depth
    prediction = torch.nn.functional.interpolate(
        predicted_depth.unsqueeze(1), size=image.size[::-1], mode="bicubic", align_corners=False,
    )
    return prediction.squeeze().cpu().numpy()

def get_dem_elevation_from_api(latitude, longitude):
    try:
        url = f"https://api.opentopodata.org/v1/eudem25m?locations={latitude},{longitude}"
        response = requests.get(url, verify=False, timeout=10)
        if response.status_code == 200:
            data = response.json()
            if data['results'] and data['results'][0]['elevation'] is not None:
                return data['results'][0]['elevation']
    except requests.exceptions.RequestException:
        return None
    return None

def get_camera_intrinsics(f_mm, s_w_mm, s_h_mm, i_w, i_h):
    fx = i_w * f_mm / s_w_mm; fy = i_h * f_mm / s_h_mm
    cx, cy = i_w / 2, i_h / 2
    return np.array([[fx, 0, cx], [0, fy, cy], [0, 0, 1]])

def get_rotation_matrix(pitch_deg, yaw_deg, roll_deg):
    yaw, pitch, roll = map(math.radians, [yaw_deg, pitch_deg, roll_deg])
    Rz = np.array([[math.cos(yaw), -math.sin(yaw), 0], [math.sin(yaw), math.cos(yaw), 0], [0, 0, 1]])
    Ry = np.array([[math.cos(pitch), 0, math.sin(pitch)], [0, 1, 0], [-math.sin(pitch), 0, math.cos(pitch)]])
    Rx = np.array([[1, 0, 0], [0, math.cos(roll), -math.sin(roll)], [0, math.sin(roll), math.cos(roll)]])
    R_gimbal = Rz @ Ry @ Rx
    R_cam_to_body = np.array([[0, 1, 0], [0, 0, 1], [1, 0, 0]]).T
    return R_gimbal @ R_cam_to_body

def calculate_destination_gps(origin_lat, origin_lon, east_m, north_m):
    bearing = math.degrees(math.atan2(east_m, north_m))
    distance_meters = math.hypot(east_m, north_m)
    destination = geodesic(meters=distance_meters).destination(Point(origin_lat, origin_lon), bearing)
    return destination.latitude, destination.longitude

def image_point_to_gps_from_depth(u, v, K, R, o_lat, o_lon, abs_depth_map):
    v_idx, u_idx = int(round(v)), int(round(u))
    if not (0 <= v_idx < abs_depth_map.shape[0] and 0 <= u_idx < abs_depth_map.shape[1]):
        return None, None
    distance_to_target = abs_depth_map[v_idx, u_idx]
    K_inv = np.linalg.inv(K)
    ray_cam = K_inv @ np.array([u, v, 1])
    ray_cam_unit = ray_cam / np.linalg.norm(ray_cam)
    point_in_cam_coords = ray_cam_unit * distance_to_target
    ned_offsets = R @ point_in_cam_coords
    ned_n, ned_e = ned_offsets[0], ned_offsets[1]
    return calculate_destination_gps(o_lat, o_lon, ned_e, ned_n)

# --- UPDATED: Function to draw overlays on the frame ---
def draw_overlays(frame, tracked_objects_data):
    """Draws bounding boxes and text for each tracked object."""
    for data in tracked_objects_data:
        x1, y1, x2, y2 = data['box']
        obj_id = data['id']
        conf = data['conf']

        # Draw the bounding box
        cv2.rectangle(frame, (x1, y1), (x2, y2), (128, 0, 128), 2) # Purple color

        # Prepare the text
        text = f"ID: {obj_id} | Conf: {conf:.2f}"
        if 'gps' in data:
            lat, lon = data['gps']
            text += f" | GPS: {lat:.5f}, {lon:.5f}"

        # Draw the text label
        cv2.putText(frame, text, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (128, 0, 128), 2)

    return frame

# --- Functions to Save Final CSV Outputs (Unchanged) ---
def save_frame_by_frame_log(all_results):
    filepath = os.path.join(OUTPUT_DIR, 'frame_by_frame_log.csv')
    with open(filepath, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['Frame', 'Timestamp', 'ObjectID', 'Latitude', 'Longitude', 'Confidence'])
        for result in all_results:
            writer.writerow([
                result['frame'], result['timestamp'], result['id'],
                f"{result['lat']:.6f}", f"{result['lon']:.6f}", f"{result['conf']:.4f}"
            ])
    print(f"✅ Frame-by-frame log saved to: {filepath}")

def save_unique_objects_summary(all_results):
    filepath = os.path.join(OUTPUT_DIR, 'unique_objects_first_seen.csv')
    first_seen = {}
    for result in all_results:
        obj_id = result['id']
        if obj_id not in first_seen:
            first_seen[obj_id] = {
                'id': obj_id, 'timestamp': result['timestamp'],
                'lat': result['lat'], 'lon': result['lon'], 'conf': result['conf']
            }
    with open(filepath, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['ObjectID', 'TimestampFirstSeen', 'Latitude', 'Longitude', 'Confidence'])
        for obj_id in sorted(first_seen.keys()):
            data = first_seen[obj_id]
            writer.writerow([
                data['id'], data['timestamp'], f"{data['lat']:.6f}",
                f"{data['lon']:.6f}", f"{data['conf']:.4f}"
            ])
    print(f"✅ Unique objects summary saved to: {filepath}")


INFO: Using Sensor Size 17.3mm x 13.0 (Mavic 3 Pro Main Cam).
INFO: Using device: cpu for deep learning models.


---
## 3. Main Execution Block

In [18]:
def main_video_pipeline(video_path, srt_path, model_path, tracker_config):
    # --- NEW: Start the timer ---
    start_time = time.time()

    print("\n--- 🚀 Starting Video Processing Pipeline 🚀 ---")

    # 1. Load Models and Parsers
    try:
        # Use the model_path passed directly to the function
        yolo_model = YOLO(model_path)
        depth_processor = AutoImageProcessor.from_pretrained(DEPTH_MODEL_NAME)
        depth_model = AutoModelForDepthEstimation.from_pretrained(DEPTH_MODEL_NAME).to(DEVICE)
        srt_metadata = parse_srt_file(srt_path)
    except Exception as e:
        print(f"❌ FATAL ERROR during initialization: {e}")
        return

    # 2. Open Video and Get Properties
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"❌ ERROR: Cannot open video file {video_path}")
        return

    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f"INFO: Video Properties: {frame_width}x{frame_height} @ {fps:.2f} FPS, {total_frames} total frames.")

    # Initialize Video Writer
    output_video_path = os.path.join(OUTPUT_DIR, 'annotated_video.mp4')
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out_video = cv2.VideoWriter(output_video_path, fourcc, fps, (frame_width, frame_height))
    print(f"INFO: Output video will be saved to: {output_video_path}")

    # 3. Process Video Frame by Frame
    frame_count = 0
    all_frame_results = []

    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            break

        frame_count += 1
        timestamp = frame_count / fps

        print(f"\n--- Processing Frame {frame_count}/{total_frames} (Timestamp: {timestamp:.2f}s) ---")

        if frame_count not in srt_metadata:
            print(f"⚠️ WARNING: No metadata in SRT for frame {frame_count}. Writing original frame.")
            out_video.write(frame)
            continue
        meta = srt_metadata[frame_count]

        # Use model.track() for detection and tracking, similar to your example
        results = yolo_model.track(frame, persist=True, tracker=tracker_config, conf=0.5)

        tracked_objects_data = []
        # Accessing tracking results
        for result in results:
            if result.boxes.id is not None:
                boxes = result.boxes.xyxy.cpu().numpy().astype(int)
                ids = result.boxes.id.cpu().numpy().astype(int)
                confs = result.boxes.conf.cpu().numpy()

                for i in range(len(ids)):
                    tracked_objects_data.append({
                        'id': ids[i],
                        'box': boxes[i],
                        'conf': confs[i]
                    })

        if not tracked_objects_data:
            print("INFO: No objects to geolocate in this frame.")
            out_video.write(frame)
            continue

        print(f"INFO: Tracking {len(tracked_objects_data)} objects.")

        try:
            # Geolocation Pipeline
            ground_elevation = get_dem_elevation_from_api(meta['latitude'], meta['longitude'])
            base_agl = (meta['abs_alt'] - ground_elevation) if ground_elevation is not None else meta['rel_alt']

            relative_depth_map = get_depth_map(frame, depth_model, depth_processor)
            scale_factor = base_agl / np.mean(relative_depth_map)
            absolute_depth_map = relative_depth_map * scale_factor

            K = get_camera_intrinsics(meta['focal_len'], SENSOR_WIDTH_MM, SENSOR_HEIGHT_MM, frame_width, frame_height)
            R = get_rotation_matrix(meta['gb_pitch'], meta['gb_yaw'], meta['gb_roll'])

            # Calculate GPS for each tracked object
            for obj_data in tracked_objects_data:
                box = obj_data['box']
                center_u = (box[0] + box[2]) / 2
                center_v = (box[1] + box[3]) / 2

                lat, lon = image_point_to_gps_from_depth(center_u, center_v, K, R, meta['latitude'], meta['longitude'], absolute_depth_map)

                if lat is not None and lon is not None:
                    print(f"  -> Geolocated Object ID {obj_data['id']} at ({lat:.6f}, {lon:.6f})")
                    obj_data['gps'] = (lat, lon) # Add GPS data to the dictionary
                    all_frame_results.append({
                        'frame': frame_count, 'timestamp': f"{timestamp:.3f}", 'id': obj_data['id'],
                        'lat': lat, 'lon': lon, 'conf': obj_data['conf']
                    })

            # Draw overlays on the frame
            annotated_frame = draw_overlays(frame.copy(), tracked_objects_data)
            out_video.write(annotated_frame)

        except Exception as e:
            print(f"❌ ERROR processing frame {frame_count}: {e}")
            traceback.print_exc()
            out_video.write(frame)
            continue

    # 4. Finalize and Save
    cap.release()
    out_video.release()
    print("\n\n--- ✅ Video Processing Complete ---")

    if all_frame_results:
        save_frame_by_frame_log(all_results)
        save_unique_objects_summary(all_results)
        print(f"✅ Annotated video saved to: {output_video_path}")
        print("\nNOTE: The output video is silent. To add the original audio back, you can use a tool like FFmpeg:")
        print(f"ffmpeg -i {output_video_path} -i {video_path} -c:v copy -c:a aac -map 0:v:0 -map 1:a:0 final_video_with_audio.mp4")
    else:
        print("INFO: No objects were successfully geolocated in the video.")

    # --- NEW: End the timer and calculate elapsed time ---
    end_time = time.time()
    elapsed_time = end_time - start_time
    print(f"\nTotal process completed in {elapsed_time:.2f} seconds.")
    if total_frames > 0:
        print(f"Average time per frame: {elapsed_time / total_frames:.3f} seconds.")


# --- Run the main pipeline if files were uploaded ---
# The check now includes model_path to be safe, and it is passed to the function
if 'video_path' in locals() and video_path and 'srt_path' in locals() and srt_path and 'model_path' in locals() and model_path and 'tracker_config_path' in locals() and tracker_config_path:
    main_video_pipeline(video_path, srt_path, model_path, tracker_config_path)
else:
    print("\n❌ Please run Cell 1 to upload all required files before running this cell.")


--- 🚀 Starting Video Processing Pipeline 🚀 ---
INFO: Parsing SRT file: DJI_20250618120033_0001_D (2).SRT
✅ Successfully parsed metadata for 1931 frames from SRT.
INFO: Video Properties: 1920x1080 @ 29.97 FPS, 1931 total frames.
INFO: Output video will be saved to: Video_Processing_Output/annotated_video.mp4

--- Processing Frame 1/1931 (Timestamp: 0.03s) ---



AttributeError: 
            'IterableSimpleNamespace' object has no attribute 'model'. This may be caused by a modified or out of date ultralytics
            'default.yaml' file.
Please update your code with 'pip install -U ultralytics' and if necessary replace
            /usr/local/lib/python3.11/dist-packages/ultralytics/cfg/default.yaml with the latest version from
            https://github.com/ultralytics/ultralytics/blob/main/ultralytics/cfg/default.yaml
            

---
## 5. View Sampling Distribution Distances